# Hook demo (facilitator only)
### Agenda item 1, 00:00-00:08 -- live skit, record, press one button, show minutes

**This notebook is for whoever is running the hook live, not for participants** -- it has the
microphone recorder and the one-button demo, nothing else. The participants' notebook
(`audio_workshop.ipynb`) has its own short cell that points here.

Run cell 1 **before the session starts** (it downloads and loads the Whisper model, which takes a
minute or two -- don't do this live in front of the room). Then during the hook: run cell 2 to
record the skit from the browser microphone, cell 3 to press the button on what was just recorded,
and cell 4 if the recording fails and the backup WAV is needed instead.

See `demo/FACILITATOR.md` for what to say and when.

## 1. Setup -- run this before the session (preloads Whisper)
On Colab: **Runtime > Change runtime type > T4 GPU** makes this faster. Upload `workshop_files.zip`
to the Files panel (left) first.

In [ ]:
import os, sys, shutil, subprocess, importlib.util, zipfile
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

if importlib.util.find_spec("whisper") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai-whisper"], check=True)
if os.path.exists("workshop_files.zip") and not os.path.exists("prototype"):
    zipfile.ZipFile("workshop_files.zip").extractall(".")
PROTO = next((p for p in ("prototype", "../prototype") if os.path.isdir(p)), None)
PROTO or sys.exit("Upload workshop_files.zip to the Files panel (left), then run this cell again.")
sys.path.insert(0, PROTO)
print("ffmpeg:", "found" if shutil.which("ffmpeg") else "NOT FOUND")

# PRELOAD medium.en now, not during the hook -- the first call to Whisper downloads
# the model (~1.5 GB) and loads it, which is slow; do that here, ahead of time.
import whisper
import transcriber
transcriber._model = whisper.load_model(transcriber.WHISPER_MODEL)
print(f"Whisper ({transcriber.WHISPER_MODEL}) preloaded and ready.")


## 2. Record the skit from the browser microphone
Click record, do the skit (two presenters, `demo/script.txt`), click stop. Saves to `skit.wav`.
Needs Colab (uses `google.colab.output`); on a laptop, record with any app instead and skip to cell 3
with that file's path.

In [ ]:
from google.colab import output
from google.colab.output import eval_js
from base64 import b64decode
from IPython.display import Javascript, display
import subprocess

RECORD_JS = """
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = () => resolve(reader.result);
  reader.readAsDataURL(blob);
});
var recorder, chunks = [];
async function startRecording() {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  recorder = new MediaRecorder(stream);
  chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
}
async function stopRecording() {
  recorder.stop();
  await new Promise(resolve => recorder.onstop = resolve);
  const blob = new Blob(chunks);
  return await b2text(blob);
}
"""

def record_skit(out_path: str = "skit.wav") -> str:
    """Browser mic -> out_path (WAV). Click to start, run the next cell's
    input() prompt (or just wait for it) to stop -- two calls, not a
    widget, since that is more reliable across Colab versions."""
    display(Javascript(RECORD_JS))
    eval_js("startRecording()")
    input("Recording... press Enter here when the skit is done. ")
    data_url = eval_js("stopRecording()")
    _, encoded = data_url.split(",", 1)
    webm_path = out_path.replace(".wav", ".webm")
    with open(webm_path, "wb") as f:
        f.write(b64decode(encoded))
    # ffmpeg converts the browser's webm/opus to WAV -- Whisper reads WAV directly.
    subprocess.run(["ffmpeg", "-y", "-i", webm_path, out_path], check=True, capture_output=True)
    print(f"Saved {out_path}")
    return out_path

skit_path = record_skit()


## 3. Press. One. Button. (the live recording)

In [ ]:
from demo import run_demo
from IPython.display import Markdown, display

minutes = run_demo(skit_path)
display(Markdown(minutes))


## 4. Backup: if the recording didn't work, use the two-voice backup WAV instead
`demo/backup_meeting.wav` -- see `demo/make_backup_wav.py` for how it was made, and
`demo/FACILITATOR.md` for when to use it.

In [ ]:
backup_path = next(p for p in ("demo/backup_meeting.wav", "../demo/backup_meeting.wav") if os.path.isfile(p))
minutes = run_demo(backup_path)
display(Markdown(minutes))
